# Lab 4: Writing the Description Prompt

**Workshop 2 · Block 3 · 20 minutes**

Whatever the description does not mention cannot be retrieved. The prompt you write here sets the ceiling on your Level 3 and Level 5 scores.

**What you will do**

1. Write your description prompt
2. Run it on the five photos and **read the output before indexing**
3. Revise and run again (most teams need two or three goes)
4. Save the descriptions, index them with metadata, and re-score
5. Swap prompts with the team next to you

Step 2 is the one people skip. If you cannot answer the test questions from the descriptions yourself, your index cannot either.

> **You need:** the `images/` folder with the five photos. The lab 2 index is used if you have it; if not, one is built for you.
>
> **You leave with:** `data/descriptions.json`, and five image chunks in `data/chroma` tagged with `kind`, `type` and `zone`. **Lab 5 counts these**, so finish Step 4.

> **How to use this notebook**
>
> 1. Run the cells **in order, top to bottom** (Shift+Enter).
>    `# ▶ RUN` cells just run. `# ✏️ EDIT` cells need you to change something first.
> 2. Most steps end with a **✅ Check** cell. If it prints ❌, fix that before moving on. The message says how.
> 3. If the kernel restarts, run the three **Setup** cells again, then carry on from where you were.
> 4. **Turn your VPN off.** The gateway rejects VPN addresses with a 403.

---
## Setup (2 minutes)

Three cells. Run all three before anything else. They are the same in labs 3, 4 and 5, so each notebook works on its own.

Your key is asked for with `getpass`: it is not shown, not saved, and gone when the kernel stops. If you have a `.env` file with `AZURE_OPENAI_KEY` in it, you will not be asked. **Never commit a notebook with a key visible in its output.**

In [105]:
# ▶ RUN  Setup 1 of 3: install the packages into THIS notebook's Python.
# Quick if they are already there. If it installs anything new, restart the
# kernel (Kernel > Restart) and carry on from Setup 2.
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [106]:
# ▶ RUN  Setup 2 of 3: connect to the gateway and test all three routes.
import getpass, json, os, re, time, statistics
from pathlib import Path
from openai import AzureOpenAI

try:                                  # optional .env, see .env.example
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_BASE   = os.getenv("AZURE_CHAT_BASE",  "https://api-iw.azure-api.net/sig-shared-jpeast-increased")
EMBED_BASE  = os.getenv("AZURE_EMBED_BASE", "https://api-iw.azure-api.net/sig-embedding")

CHAT_DEPLOYMENT   = os.getenv("CHAT_DEPLOYMENT",   "gpt-4o-mini")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT", "gpt-5-mini")
EMBED_DEPLOYMENT  = os.getenv("EMBED_DEPLOYMENT",  "text-embedding-3-small")

# The gateway takes the FULL path as the endpoint, so each client is bound
# to one deployment. The chat route has no /openai segment and the
# embedding route does. That is correct: do not make them match.
CHAT_URL   = f"{CHAT_BASE}/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
VISION_URL = f"{CHAT_BASE}/deployments/{VISION_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
EMBED_URL  = f"{EMBED_BASE}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version={API_VERSION}"

# .strip() matters: a pasted key often carries a trailing newline -> 401.
KEY = (os.getenv("AZURE_OPENAI_KEY") or getpass.getpass("Azure OpenAI key: ")).strip()


def _client(url):
    return AzureOpenAI(azure_endpoint=url, api_key=KEY, api_version=API_VERSION)


chat_client   = _client(CHAT_URL)
vision_client = _client(VISION_URL)
embed_client  = _client(EMBED_URL)

failed = []
for label, fn in [
    ("chat  ", lambda: chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("vision", lambda: vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("embed ", lambda: f"{len(embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=['test']).data[0].embedding)} dimensions"),
]:
    try:
        print(f"{label}  OK      {fn()}")
    except Exception as e:
        failed.append(str(e))
        print(f"{label}  FAILED  {type(e).__name__}: {str(e)[:110]}")

if not failed:
    print("\n✅ Connected. Carry on.")
elif any("403" in f for f in failed):
    print("\n❌ 403 Forbidden: your VPN is on. Turn it off and run this cell again.")
elif any("401" in f for f in failed):
    print("\n❌ 401: the key is wrong. Run this cell again and paste it carefully.")
else:
    print("\n❌ Not connected. 404 means the URL is wrong (did you edit this cell?). Ask a mentor.")

chat    OK      Linked
vision  OK      connected
embed   OK      1536 dimensions

✅ Connected. Carry on.


In [107]:
# ▶ RUN  Setup 3 of 3: helper functions. Nothing to change here.

def chat(messages, model=None, temperature=0.0, max_tokens=512):
    r = chat_client.chat.completions.create(
        model=model or CHAT_DEPLOYMENT, messages=messages,
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""


def ask(prompt, system=None, **kw):
    msgs = ([{"role": "system", "content": system}] if system else [])
    return chat(msgs + [{"role": "user", "content": prompt}], **kw)


def embed(texts, batch_size=256):
    """Embed a LIST of strings in batches (one call per string is slow)."""
    texts = [t.replace("\n", " ") for t in texts]
    out = []
    for i in range(0, len(texts), batch_size):
        r = embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


def chunk(text, size=800, overlap=100):
    text = " ".join(text.split())
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]


# One PersistentClient per folder for the life of the kernel. Opening a
# second one on the same folder corrupts the connection ("attempt to write
# a readonly database"), so rebuilds reset the COLLECTION, not the folder.
_stores = {}


def get_store(path="data/chroma", name="workshop", reset=False):
    import chromadb, shutil
    if path not in _stores:
        if reset and Path(path).exists():
            shutil.rmtree(path)
        try:
            _stores[path] = chromadb.PersistentClient(path=path)
        except KeyError as e:
            raise RuntimeError(
                f"chromadb cannot read the index at {path} ({e}). It was built by a "
                f"different chromadb version. Delete that folder and run again.") from None
    client = _stores[path]
    if reset:
        try:
            client.delete_collection(name)
        except Exception:
            pass
    return client.get_or_create_collection(name)


def ensure_index(path="data/chroma", name="workshop"):
    """Open the index lab 2 built. If it is missing or empty, build it from
    sample_corpus/ so this notebook still works if you skipped lab 2."""
    store = get_store(path, name)
    if store.count() == 0:
        print("No lab 2 index found, so building one from sample_corpus/ (about 30 seconds)...")
        texts, metas = [], []
        for f in sorted(Path("sample_corpus").glob("*.txt")):
            for i, piece in enumerate(chunk(f.read_text(encoding="utf-8"))):
                texts.append(piece)
                metas.append({"url": f.name, "position": i, "kind": "text"})
        store = get_store(path, name, reset=True)
        add_to_store(store, texts, metas)
    return store


def add_to_store(store, texts, metadatas, ids=None, batch_size=128):
    ids = ids or [f"c{i}" for i in range(len(texts))]
    for i in range(0, len(texts), batch_size):
        sl = slice(i, i + batch_size)
        store.add(ids=ids[sl], documents=texts[sl],
                  embeddings=embed(texts[sl]), metadatas=metadatas[sl])


def query(store, question, k=5, where=None):
    """The k nearest chunks, closest first. Distance is squared L2: lower is closer.
    `where` filters on metadata first. One field: {"kind": "image"}.
    Two or more fields need $and: {"$and": [{"year": "2025"}, {"type": "poster"}]}."""
    r = store.query(query_embeddings=embed([question]), n_results=k, where=where or None)
    return [{"text": d, "metadata": m, "distance": dist}
            for d, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]


def show(chunks, chars=200):
    if not chunks:
        print("  (nothing returned)")
        return
    for c in chunks:
        print(f"  {c['distance']:.3f}  {c['metadata'].get('url', '?')}")
        print(f"         {c['text'][:chars].strip()}\n")


def timed(fn, *a, **kw):
    t0 = time.time()
    return fn(*a, **kw), time.time() - t0


def normalise(s):
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())


def answer_present(expected, chunks):
    """Is the expected answer anywhere in the retrieved text? A crude string
    match: read the chunks before you believe a FOUND."""
    hay  = normalise(" ".join(c["text"] for c in chunks))
    need = normalise(expected).strip()
    if need and need in hay:
        return True
    terms = [t for t in need.split() if len(t) > 3]
    return bool(terms) and sum(t in hay for t in terms) / len(terms) >= 0.8


def load_dev_set(path="dev_set.json"):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def describe_image(image_path, prompt, model=None):
    """One image plus an instruction, sent to the vision deployment.
    There is deliberately no default prompt: writing it is lab 4."""
    import base64, mimetypes
    p    = Path(image_path)
    mime = mimetypes.guess_type(p.name)[0] or "image/jpeg"
    b64  = base64.b64encode(p.read_bytes()).decode()
    r = vision_client.chat.completions.create(
        model=model or VISION_DEPLOYMENT,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        ]}])
    return r.choices[0].message.content or ""


# Where each workshop photo was taken. Lab 4 stores this as metadata and
# lab 5 filters on it.
PHOTO_ZONES = {
    "Brainstorming aerial.JPG": "Brainstorming Area",
    "Makerspace close up.JPG":  "Makerspace A",
    "makerspace.JPG":           "Makerspace A",
    "Open Event .JPG":          "Open Event Area",
    "Open event.JPG":           "Open Event Area",
}

print("✅ helpers loaded")

✅ helpers loaded


---
## The five fields to ask for

| # | Field | Ask for | Why |
|---|---|---|---|
| 1 | Location | The room or area, using the name on the signage | A question naming a room can reach the image |
| 2 | Objects, with counts | "Eleven chairs", never "some chairs" | Counting questions need numbers |
| 3 | Visible text, word for word | Every sign, poster and label, copied exactly | Questions ask for exact words |
| 4 | Spatial relations | "Glass doors directly to the right of the seating" | Questions ask what is next to what |
| 5 | Colours and materials | Only where they identify something | Tells similar rooms apart |

**Field 3 is the one to get right.** Tell the model to copy text exactly rather than summarise, and to write `[unreadable]` rather than guess.

The next cell loads the photos and three **test questions**, one each for counts, text and layout. Each answer is visible in a photo and appears nowhere in the text corpus, so only a good description can answer it. (An earlier test run read "RAG" on the screen as "PAG": the text question is a real trap.)

In [108]:
# ▶ RUN
IMAGES = sorted(str(p) for p in Path("images").iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
assert len(IMAGES) >= 5, f"Found {len(IMAGES)} photos in images/. This lab needs the five provided photos."

TEST_QUESTIONS = [
    {"field": "count",  "question": "How many cushions are on the long low bench in front of the tiered seating in the brainstorming area?", "answer": "3"},
    {"field": "text",   "question": "What is the title of the slide on the large screen in the makerspace?", "answer": "Building RAG with Image Data"},
    {"field": "layout", "question": "What is directly to the right of the tiered seating in the brainstorming area?", "answer": "glass doors"},
]

for i, p in enumerate(IMAGES):
    print(f"  [{i}] {Path(p).name:<26} zone: {PHOTO_ZONES.get(Path(p).name, '?')}")

  [0] Brainstorming aerial.JPG   zone: Brainstorming Area
  [1] Makerspace close up.JPG    zone: Makerspace A
  [2] Open Event .JPG            zone: Open Event Area
  [3] Open event.JPG             zone: Open Event Area
  [4] makerspace.JPG             zone: Makerspace A


---
## Step 1 of 5 · Write your prompt (4 minutes)

Write **instructions**, not a request for a description. First look at the two examples, then write your own in the cell after.

In [109]:
# ▶ RUN  (two example prompts to compare in Step 2)
PROMPT_A = "Describe this image."

PROMPT_B = """List, in this order:
1. the room or area name shown on any sign
2. every object present, with a count
3. all visible text, copied exactly as written
4. what faces what, and what is next to what
5. colours and materials, only where distinctive

Write [unreadable] for text you cannot make out.
Do not summarise. Do not add anything not visible in the image."""
print("examples loaded")

examples loaded


In [110]:
# ✏️ EDIT: replace the TODO line with your own prompt, then run.
# Checklist: does it ask, by name, for all five fields? Does it say to copy
# text exactly and write [unreadable] instead of guessing?
DESCRIPTION_PROMPT = """You are describing one photograph of a zone inside the HKU
Tam Wing Fan Innovation Wing. The description will be stored and searched later, so
it must be factual, specific and self-contained.

Report only what is visible in this image. Never state a fact you cannot see, and
never give a number you have not counted.

Style rules:
- Plain declarative sentences. No hedging: no "appears to be", "likely",
  "consistent with", "possibly". No confidence levels.
- Do not describe lighting, mood, atmosphere, or where the camera is standing.
- No preamble and no closing summary. Start with the Zone line.

Zone: name the area on a single line at the top, choosing from this list:
Makerspace Assembly Area, Machine Shop, 3D Printing Lab, Laser Cutting Area,
Electronics Workbenches, Spray Paint Station, Thematic Workshop, CAD Studio,
AR-VR Studio, Podcast Studio, Dream Chamber, Brainstorming Area, Poster Hallway,
Meeting Room. Pick one only. Do not use the wording on a sign as the zone name,
and do not repeat the zone name in the sections below.

1. TEXT
Copy every readable word in the image exactly as written: spelling, capitalisation,
punctuation and any Chinese characters unchanged. Do not translate, correct or
summarise. Give each one's position, for example: the sign above the door reads
"MACHINE SHOP". Include signage, door labels, machine model names, brand names,
safety notices, posters, slides, screens, whiteboards and equipment labels. Write
[unreadable] for anything you cannot make out. Never guess at a partly visible word.

2. OBJECTS AND COUNTS
For any group of similar items, list them one by one from left to right with their
colour and position, then give the total on a separate line. Count what is actually
in the frame. Do not assume rows are equal length, that a group forms a grid, or
that colours are grouped by row. If a group is partly out of frame or occluded, say
so and give the number you can see. Cover furniture, machines, printers, screens,
tools, projects in progress and people.

3. LAYOUT
State what is next to, in front of, behind, above or facing what.

4. MATERIALS AND COLOURS
Only where they identify a specific machine, surface or partition.

5. ACTIVITY
What machines and people are doing, stated plainly. Do not speculate about who
anyone is."""

---
## Step 2 of 5 · Run it and read the output (6 minutes)

First the two example prompts on the same photo, so you can see what the prompt alone changes. Then your prompt on all five photos, one cell each. Each call takes **10 to 30 seconds**.

> **Output cut off?** Long output gets hidden. **VS Code:** click *"View as a scrollable element"* under the cell. **JupyterLab:** right-click the output and choose *"Enable Scrolling for Outputs"*. Do not skip the reading: it is the whole of this step.

In [111]:
# ▶ RUN  example A on the first photo
print("PROMPT A:", PROMPT_A, "\n" + "-" * 60)
print(describe_image(IMAGES[0], PROMPT_A))

PROMPT A: Describe this image. 
------------------------------------------------------------
The photo shows a bright, modern indoor collaboration or lounge area with tiered seating and colorful modular furniture. In the center-left are low square ottomans arranged in rows — red, orange and yellow cubes — facing a stepped platform of wide bleacher-style tiers with built-in electrical outlets and dark-blue cushions. A long bench with teal cushions runs along the front of the platforms. Above the steps a slanted blue sign reads “ENGINEERING INNOVATION.” To the right are glass doors and to the left a small counter or kitchenette area with cabinets, a sink and posted notices. The space has neutral carpet tiles, pale green and grey walls, and fluorescent ceiling lights; the image shows slight wide-angle distortion.


In [112]:
# ▶ RUN  example B on the same photo
print("PROMPT B\n" + "-" * 60)
print(describe_image(IMAGES[0], PROMPT_B))
print("\nSame model, same photo, same cost. Only the prompt changed.")

PROMPT B
------------------------------------------------------------
1. ENGINEERING INNOVATION

2. Every object present, with a count:
- Small square ottoman cubes: 18
- Tiered seating / stepped platform (bleacher-style): 1
- Long low bench with teal cushions (segmented): 5 cushion segments, 1 continuous bench
- Wooden bench supports under teal cushions: 5
- Built-in counter / cabinet unit along back wall: 1
- Freestanding sign/stand near left of counter: 1
- Trash bin / waste receptacle (stainless or plastic) near left of counter: 1
- Wall-mounted signs / posters above counter: 5
- Small countertop dispensers / bottles (on counter): 2
- Clear countertop box / dispenser (on counter): 1
- Glass double doors / glass partition at right: 1 set
- Ceiling recessed/rectangular fluorescent lights: 8
- Vertical decorative slats / fins on wall (green): 1 installation
- Carpet floor covering: 1
- Staircase / opening to upper level (visible at upper right): 1
- Wall-mounted small notices on glass

In [113]:
# ▶ RUN  (checks your prompt, then gets ready)
assert "TODO" not in DESCRIPTION_PROMPT, (
    "DESCRIPTION_PROMPT is still the placeholder. Write your prompt in Step 1 first. "
    "Otherwise the model answers the word TODO instead of describing the photo, and that is what gets indexed.")
descriptions = {}
print("✅ prompt set. Run the five cells below, one per photo.")

%pip install -q pillow

✅ prompt set. Run the five cells below, one per photo.
Note: you may need to restart the kernel to use updated packages.


In [126]:
# ▶ RUN  replaces the five "photo N of 5" cells. Skips work already done.
from PIL import Image, ImageOps

TEXT_PROMPT = """Transcribe every readable word in this image and nothing else.

Copy each exactly as written: spelling, capitalisation, punctuation and any Chinese
characters unchanged. Do not translate, correct or summarise. Give each one's
position, for example: the sign above the sink reads "ATTENTION - HAND-WASHING ONLY".
Include signs, posters, notices, labels, screens, whiteboards, brand names and model
numbers. Write [unreadable] for anything you cannot make out. Never guess at a partly
visible word. If there is no text, write "no text visible"."""

MARKER = "TEXT READ CLOSE UP"


def upright(path):
    """Vision APIs ignore EXIF rotation, so save a correctly rotated copy first.
    Makerspace close up.JPG is stored upside down - without this the model reads
    the slide inverted."""
    out = Path("data/upright"); out.mkdir(parents=True, exist_ok=True)
    dst = out / Path(path).name
    ImageOps.exif_transpose(Image.open(path)).save(dst, quality=92)
    return str(dst)


def describe_tiled(path):
    """Four overlapping quarters, so small signs are near full resolution."""
    im = Image.open(path)
    W, H = im.size
    ox, oy = int(W * 0.15), int(H * 0.15)
    boxes = {
        "top left":     (0,           0,           W // 2 + ox, H // 2 + oy),
        "top right":    (W // 2 - ox, 0,           W,           H // 2 + oy),
        "bottom left":  (0,           H // 2 - oy, W // 2 + ox, H),
        "bottom right": (W // 2 - ox, H // 2 - oy, W,           H),
    }
    out = Path("data/tiles"); out.mkdir(parents=True, exist_ok=True)
    parts = []
    for name, box in boxes.items():
        tile = out / f"{Path(path).stem}_{name.replace(' ', '_')}.jpg"
        im.crop(box).save(tile, quality=92)
        parts.append(f"[{name} quarter] " + describe_image(str(tile), TEXT_PROMPT))
    return "\n".join(parts)


try:
    descriptions
except NameError:
    descriptions = {}

for n, p in enumerate(IMAGES, 1):
    if MARKER in descriptions.get(p, ""):
        print(f"[{n}/{len(IMAGES)}] {Path(p).name} already done, skipped")
        continue
    fixed = upright(p)
    scene = describe_image(fixed, DESCRIPTION_PROMPT)
    text = describe_tiled(fixed)
    descriptions[p] = scene + "\n\n" + MARKER + "\n" + text
    print(f"[{n}/{len(IMAGES)}] {Path(p).name} described")

print(f"\n{len(descriptions)} of {len(IMAGES)} photos described.\n")

for p, d in descriptions.items():
    print("=" * 70); print(Path(p).name); print("=" * 70); print(d, "\n")

[1/5] Brainstorming aerial.JPG already done, skipped
[2/5] Makerspace close up.JPG described
[3/5] Open Event .JPG described
[4/5] Open event.JPG described
[5/5] makerspace.JPG described

5 of 5 photos described.

Brainstorming aerial.JPG
Zone: Brainstorming Area

1. TEXT
- The blue angled wall sign above the stepped seating reads "ENGINEERING" on the top line and "INNOVATION" on the bottom line.
- The white sign on the glass door at the right reads "NOTICE".

2. OBJECTS AND COUNTS
- From left to right in the foreground there is a rectangular grid of cube ottomans arranged in four columns and three rows. Column 1 (leftmost) contains three red cubes. Column 2 contains three orange cubes. Column 3 contains three light‑orange cubes. Column 4 (rightmost) contains three yellow cubes. Total cube ottomans: 12.
- To the right of the cube ottomans and directly in front of the stepped seating there is one long low bench with five rectangular seat cushions in teal/green/blue tones mounted on ligh

In [115]:
# ▶ RUN  photo 2 of 5
p = IMAGES[1]; descriptions[p] = describe_image(p, DESCRIPTION_PROMPT)
print(Path(p).name, "\n" + "-" * 70); print(descriptions[p])

Makerspace close up.JPG 
----------------------------------------------------------------------
Zone: Brainstorming Area

TEXT:
The large central monitor near the windows reads "Building PAG with Image Data".
A smaller line on the same central monitor reads "Validation dataset".
The freestanding whiteboard at right has handwritten notes; the handwriting is [unreadable].

OBJECTS AND COUNTS:
From left to right across the image: one black tripod with a mounted camera near the left glass wall. One group of people around a central table: person wearing a light grey hoodie with their back to the camera (leftmost in the group), person with dark hair and glasses seated/leaning, person wearing a light blue shirt standing behind the seated person, person wearing a yellow-and-black shirt standing, person wearing a green T-shirt seated with their back to the camera. One person in a black T-shirt standing at the central back table near the large monitor. Total people visible: 6.
On the middle fore

In [116]:
# ▶ RUN  photo 3 of 5
p = IMAGES[2]; descriptions[p] = describe_image(p, DESCRIPTION_PROMPT)
print(Path(p).name, "\n" + "-" * 70); print(descriptions[p])

Open Event .JPG 
----------------------------------------------------------------------
Zone: Brainstorming Area

1. TEXT
- The blue panel on the left wall reads "ATTENTION".
- The blue sign on the left wall reads "NOTICE".
- A white sign on the left wall reads "KEEP CLEAN".

2. OBJECTS AND COUNTS
- Three light-grey chairs with metal legs lined against the left wall. Total: 3.
- A vertical stack of light-grey chairs on a black trolley positioned to the right of the three chairs. Total: 12.
- One light-grey chair with metal legs positioned in front of the stacked chairs. Total: 1.
- Four white triangular tables joined into a central star-shaped assembly in the middle of the room. Total triangular tables: 4.
- Four light-grey chairs with metal legs placed around the central star-shaped table assembly. Total: 4.
- One rectangular white table against the back wall with four light-grey chairs around it. Total tables: 1; total chairs at that table: 4.
- One white table on the mid-right with 

In [117]:
# ▶ RUN  photo 4 of 5
p = IMAGES[3]; descriptions[p] = describe_image(p, DESCRIPTION_PROMPT)
print(Path(p).name, "\n" + "-" * 70); print(descriptions[p])

Open event.JPG 
----------------------------------------------------------------------
Zone: Brainstorming Area

1. TEXT
- The sign on the blue panel reads "Open Event Area".
- The orange sign reads "ATTENTION".
- The blue sign reads "NOTICE".
- The white sign reads "KEEP CLEAN".

2. OBJECTS AND COUNTS
- A blue wall panel with the signs listed above at the far left. Total: 1.
- A stack of grey plastic chairs leaning against the left wall. Visible chairs in the stack: 10. Total in this group: 10.
- A curved white modular table made of three connected white segments at the front-left. Total: 1 table, 3 connected segments.
- Three individual grey chairs positioned around the curved modular table (front-left). Total: 3.
- A white pentagonal/hexagonal table near the front-center with five grey chairs around it. Total: 1 table, 5 chairs.
- A white pentagonal/hexagonal table slightly behind the front-center table with five grey chairs around it. Total: 1 table, 5 chairs.
- A white pentagonal/

In [118]:
# ▶ RUN  photo 5 of 5
p = IMAGES[4]; descriptions[p] = describe_image(p, DESCRIPTION_PROMPT)
print(Path(p).name, "\n" + "-" * 70); print(descriptions[p])

makerspace.JPG 
----------------------------------------------------------------------
Zone: Electronics Workbenches

1. TEXT
The green sign above the far doorway reads "EXIT".

2. OBJECTS AND COUNTS
- Light-coloured rectangular wooden worktable in the foreground, centre. Total: 1.
- Light-coloured rectangular wooden worktable behind the foreground table, centre-left. Total: 1.
- Light-coloured rectangular wooden worktable behind the foreground table, centre-right. Total: 1.
- Light-coloured rectangular wooden worktable at left with a seated person using a laptop. Total: 1.
- Light-coloured rectangular wooden worktable at right foreground (near bottom right corner). Total: 1.
- Light-coloured rectangular wooden worktable along the right side of the room near the windows. Total: 1.
- Light-coloured rectangular wooden worktable in the far back centre of the room. Total: 1.
- Total worktables visible: 7.

- Dark brown round-top stools with black metal legs under and around tables, listed 

In [119]:
# ✅ Check: can each test question be answered from your descriptions alone?
if len(descriptions) < len(IMAGES):
    print(f"❌ Only {len(descriptions)} of {len(IMAGES)} photos described. Run all five photo cells first.")
for q in TEST_QUESTIONS:
    print(f"\n[{q['field']}] {q['question']}\n   expected: {q['answer']}")
    if q["field"] == "count":
        # A bare digit matches almost anything, so show the relevant sentences instead.
        lines = [s.strip() for d in descriptions.values() for s in re.split(r"[\n.]", d)
                 if re.search(r"cushion|bench", s, re.I)]
        print("   ✏️ check by eye. Sentences that mention cushions or the bench:")
        for s in lines[:6]:
            print("     -", s[:120])
        if not lines:
            print("     ❌ none. Your prompt did not get the bench or its cushions described.")
    else:
        where = [Path(p).name for p, d in descriptions.items() if answer_present(q["answer"], [{"text": d}])]
        print("   ✅ found in: " + ", ".join(where) if where else "   ❌ NOWHERE. Revise your prompt (Step 3).")


[count] How many cushions are on the long low bench in front of the tiered seating in the brainstorming area?
   expected: 3
   ✏️ check by eye. Sentences that mention cushions or the bench:
     - - To the right of the cube ottomans and directly in front of the stepped seating there is one long low bench with five r
     - Total bench cushions: 5
     - - Behind the cube ottomans and bench along the left wall there is one low beige cabinet with a white countertop
     - - At the top of the stepped seating there is one small dark rectangular cushion or backrest
     - Total small cushions/backrests: 1
     - - The long bench with five cushions is directly in front of and along the base of the stepped seating and to the right o

[text] What is the title of the slide on the large screen in the makerspace?
   expected: Building RAG with Image Data
   ✅ found in: Makerspace close up.JPG

[layout] What is directly to the right of the tiered seating in the brainstorming area?
   expected: g

---
## Step 3 of 5 · Revise (3 minutes)

Most teams need two or three attempts; the first is almost always too general. Go back to **Step 1**, edit `DESCRIPTION_PROMPT`, and run **Step 2** again from the "checks your prompt" cell.

| Prompts that scored | Prompts that did not |
|---|---|
| Asked for counts explicitly | Asked to "describe this image" |
| Asked for text to be copied exactly | Asked for a summary or caption |
| Named the room or area | Did not mention text at all |
| Said what to do when unreadable | Left the model to decide what mattered |

Move on when the ✅ Check shows all three answers.

---
## Step 4 of 5 · Save, index with metadata, re-score (4 minutes)

**Save** the descriptions to `data/descriptions.json`, so you never pay to describe the same photo twice.

**Index** them into the same store as your text, as ordinary chunks. Each one also gets **metadata**, the fields a question can later filter on:

| Field | Value | Used for |
|---|---|---|
| `kind` | `"image"` | telling image chunks from text chunks |
| `type` | `"photo"` | what the item is (on the website this would be `"poster"`) |
| `zone` | e.g. `"Open Event Area"` | where it was taken |
| `url` | the photo's path | tracing an answer back to its photo |

Lab 5 counts these chunks by filtering on exactly these fields.

In [120]:
# ▶ RUN
Path("data").mkdir(exist_ok=True)
Path("data/descriptions.json").write_text(
    json.dumps({Path(p).as_posix(): d for p, d in descriptions.items()}, indent=1), encoding="utf-8")
print(f"saved {len(descriptions)} descriptions -> data/descriptions.json")

store = ensure_index()

# Remove the previous run's image chunks first. Chroma will not overwrite an
# id that already exists, so without this a revised prompt never reaches the index.
store.delete(where={"kind": "image"})

paths = list(descriptions)
add_to_store(
    store,
    texts=[descriptions[p] for p in paths],
    metadatas=[{"url": Path(p).as_posix(), "kind": "image", "type": "photo",
                "zone": PHOTO_ZONES.get(Path(p).name, "unknown")} for p in paths],
    ids=[f"img_{i}" for i in range(len(paths))],
)
print(f"indexed. The store now holds {store.count()} chunks.")

saved 5 descriptions -> data/descriptions.json
indexed. The store now holds 36 chunks.


In [121]:
# ✅ Check
got = store.get(where={"kind": "image"}, include=["metadatas"])["metadatas"]
if len(got) == len(IMAGES):
    zones = sorted({m.get("zone") for m in got})
    print(f"✅ {len(got)} image chunks indexed, with zones: {', '.join(zones)}. Lab 5 will count these.")
else:
    print(f"❌ Expected {len(IMAGES)} image chunks, found {len(got)}. Describe all five photos (Step 2), then run the cell above again.")

✅ 5 image chunks indexed, with zones: Brainstorming Area, Makerspace A, Open Event Area. Lab 5 will count these.


In [122]:
# ▶ RUN  re-score: does an image chunk now come back for each question?
dev = load_dev_set("dev_set.json")
checks = ([(q["question"], q["answer"]) for q in TEST_QUESTIONS] +
          [(d["question"], d["answer"]) for d in dev if d["level"] in (3, 5)])

for question, expected in checks:
    hits = query(store, question, k=3)
    ok = any(c["metadata"].get("kind") == "image" and answer_present(expected, [c]) for c in hits)
    print(("✅ " if ok else "   ") + question)
    print(f"   expected: {expected}")
    for c in hits:
        print(f"   {c['distance']:.3f}  {c['metadata'].get('kind', 'text'):<5}  {c['metadata'].get('url', '?')[:50]}")
    print()
print("The Lv3 poster question should still miss: that poster is on the website, not in these photos (cause A in lab 3).")

✅ How many cushions are on the long low bench in front of the tiered seating in the brainstorming area?
   expected: 3
   0.870  image  images/Open Event .JPG
   0.962  image  images/Open event.JPG
   1.009  image  images/Brainstorming aerial.JPG

✅ What is the title of the slide on the large screen in the makerspace?
   expected: Building RAG with Image Data
   0.977  image  images/Makerspace close up.JPG
   1.010  text   venues-makerspace-a.txt
   1.050  image  images/makerspace.JPG

✅ What is directly to the right of the tiered seating in the brainstorming area?
   expected: glass doors
   0.847  image  images/Open Event .JPG
   0.922  image  images/Open event.JPG
   0.968  image  images/Brainstorming aerial.JPG

   What were the winning teams in Pitch New Tech Ideas in 2025?
   expected: Smartsocks, Digitio, Gingtrolley
   1.101  text   news-2025-awards.txt
   1.151  text   events-2025-showcase.txt
   1.219  text   funding-scheme.txt

   On the wall at the back of the brainstorming

---
## Step 5 of 5 · Swap with the team next to you (3 minutes)

Paste their prompt below and run it on the photo **your** prompt did worst on. Which facts does theirs get that yours misses?

Two teams with different prompts get different Level 3 scores from identical photos. The prompt is the whole difference.

In [123]:
# ✏️ EDIT: paste their prompt, and pick the photo number (0 to 4)
THEIR_PROMPT = """
paste their prompt here
"""
PHOTO = 0

In [124]:
# ▶ RUN
p = IMAGES[PHOTO]
print("YOURS\n" + "-" * 60); print(descriptions.get(p) or describe_image(p, DESCRIPTION_PROMPT))
print("\nTHEIRS\n" + "-" * 60); print(describe_image(p, THEIR_PROMPT))

YOURS
------------------------------------------------------------
Zone: Brainstorming Area

1. TEXT
- The blue angled wall sign above the stepped seating reads "ENGINEERING" on the top line and "INNOVATION" on the bottom line.
- The white sign on the glass door at the right reads "NOTICE".

2. OBJECTS AND COUNTS
- From left to right in the foreground there is a rectangular grid of cube ottomans arranged in four columns and three rows. Column 1 (leftmost) contains three red cubes. Column 2 contains three orange cubes. Column 3 contains three light‑orange cubes. Column 4 (rightmost) contains three yellow cubes. Total cube ottomans: 12.
- To the right of the cube ottomans and directly in front of the stepped seating there is one long low bench with five rectangular seat cushions in teal/green/blue tones mounted on light wood supports. Total bench cushions: 5.
- Behind the cube ottomans and bench along the left wall there is one low beige cabinet with a white countertop. Total low cabinet

---
## Going further

- **Two prompts per image.** One for objects and layout, one for text only. The model attends to what you emphasise.
- **Split large images.** A poster shrunk to fit loses its small print. Cut it into overlapping tiles and describe each.
- **Run OCR alongside.** A vision model paraphrases; an OCR engine copies character by character. Tesseract and PaddleOCR run locally, so both are allowed.

## What to take to your own repository

Put your prompt into `DESCRIPTION_PROMPT` in `build/images.py`, and add `type`, `year` and (for your own photos) `zone` to the metadata in `index_descriptions()`. The template stores only `url`, `image` and `kind`.

**Next: lab 5.**